# 최종 앙상블 파이프라인

이 공개본은 ConvNeXt-Tiny·EfficientNet-B0·ResNet-50을 Stratified K-Fold로 학습하고, TTA 및 F1 가중 soft voting을 적용하는 최종 제출용 파이프라인을 대표 성과물로 제시합니다. GPU가 없을 때는 경량 설정으로 동작합니다.

저장된 실행 기록에는 검증 수치 출력이 남아 있지 않습니다. 따라서 이 노트북만으로 특정 성능 수치를 주장하지 않습니다. 아래에 남아 있는 초기 설명은 이전 베이스라인 안내이며, 실제 실행 코드는 STEP 1 이후의 앙상블 설정을 기준으로 확인해 주세요.

# 미니 경진대회 - 선박 도장 품질 분류 (최소 수정 개선본)

기존 베이스라인의 전체 흐름은 최대한 유지했습니다.

**주요 변경점은 4개뿐입니다.**
1. 흑백 128×128 픽셀을 그대로 펼치는 대신 **RGB 이미지 + 사전학습 ResNet18 특징(feature)** 사용
2. 기존처럼 **LogisticRegression**으로 최종 분류
3. 클래스 불균형을 고려해 `class_weight="balanced"` 적용
4. Accuracy와 함께 대회 지표인 **Macro F1 Score**를 기준으로 가장 좋은 `C` 값을 선택

즉, 폴더 구조 / 라벨 인코딩 / train-test split / LogisticRegression / submission.csv 생성 방식은 그대로 유지하고, **이미지 특징 추출 부분만 강화**했습니다.

> 최초 실행 시 ResNet18 사전학습 가중치 다운로드가 1회 필요할 수 있습니다. GPU가 있으면 자동으로 GPU를 사용합니다.

In [1]:
# 필요한 라이브러리가 없는 경우에만 아래 주석을 해제하여 실행하세요.
# !pip install torch torchvision scikit-learn pandas pillow


In [ ]:
# ===== STEP 1 : 설정 =====
import os, gc, math, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms as T
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything()

ROOT   = Path("/mnt/elice/dataset")
SUBMIT_DIR = ROOT / "제출용 데이터"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_GPU = DEVICE.type == "cuda"

IMG_SIZE   = 224
BATCH_SIZE = 16 if USE_GPU else 8
EPOCHS     = 14 if USE_GPU else 6
N_FOLDS    = 5  if USE_GPU else 2
LR         = 3e-4
WD         = 1e-4
LABEL_SMOOTH = 0.05
MIXUP_P    = 0.5
MIXUP_ALPHA= 0.4
NUM_WORKERS= 2 if USE_GPU else 0
AMP        = USE_GPU

# 앙상블에 쓸 백본 (GPU 없으면 1개만)
MODEL_NAMES = ["convnext_tiny", "efficientnet_b0", "resnet50"] if USE_GPU else ["efficientnet_b0"]

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
CKPT_DIR = Path("./ckpt"); CKPT_DIR.mkdir(exist_ok=True)

print("device:", DEVICE, "| folds:", N_FOLDS, "| epochs:", EPOCHS, "| models:", MODEL_NAMES)


## 데이터 불러오기

기존 폴더명과 라벨 구조는 그대로 사용합니다.

* TS_도막 손상_도막떨어짐 → 도막떨어짐
* TS_도막 손상_스크래치 → 스크래치
* TS_도장 불량_부풀음 → 부풀음
* TS_도장 불량_이물질포함 → 이물질포함
* TS_양품_선수 → 양품
* TS_양품_외판 → 양품


## ResNet18 특징 추출

기존 코드에서 가장 큰 약점은 아래 방식이었습니다.

```python
convert("L") -> 128x128 -> ravel()
```

이 방법은 색상과 공간적 패턴을 많이 잃습니다. 여기서는 **ImageNet으로 사전학습된 ResNet18**을 이용해 각 이미지를 512차원의 특징 벡터로 바꿉니다.

중요한 점은 ResNet18 전체를 새로 학습하는 것이 아니라 **특징 추출기로만 사용**하기 때문에, 완전한 딥러닝 학습보다 훨씬 가볍고 기존 LogisticRegression 구조도 유지할 수 있다는 점입니다.

## 모델 학습

학습/검증 분할은 기존처럼 `stratify`를 사용합니다. 차이점은 **분할 후 각각의 이미지를 ResNet18 특징으로 변환**한다는 점입니다.

`C` 값 몇 개만 가볍게 비교해서 **Macro F1이 가장 높은 LogisticRegression 모델**을 자동 선택합니다.

In [ ]:
# ===== STEP 2 : 데이터 수집 =====
folder_to_label = {
    "TS_도막 손상_스크래치":   "스크래치",
    "TS_도장 불량_부풀음":     "부풀음",
    "TS_도막 손상_도막떨어짐": "도막떨어짐",
    "TS_도장 불량_이물질포함": "이물질포함",
    "TS_양품_선수":            "양품",
    "TS_양품_외판":            "양품",
}

image_paths, labels = [], []
for folder, label in folder_to_label.items():
    d = ROOT / folder
    if not d.exists():
        print("[경고] 폴더 없음:", d); continue
    for p in sorted(d.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            image_paths.append(str(p)); labels.append(label)

# 손상 파일 제거
valid_p, valid_l = [], []
for p, l in zip(image_paths, labels):
    try:
        with Image.open(p) as im: im.verify()
        valid_p.append(p); valid_l.append(l)
    except Exception:
        print("[손상 파일 제외]", p)
image_paths, labels = np.array(valid_p), np.array(valid_l)

le = LabelEncoder()
Y  = le.fit_transform(labels)
NUM_CLASSES = len(le.classes_)

print("총 이미지:", len(image_paths))
print("클래스:", list(le.classes_))
print(pd.Series(labels).value_counts())

submit_paths = sorted([str(p) for p in SUBMIT_DIR.rglob("*")
                       if p.is_file() and p.suffix.lower() in IMG_EXTS])
print("제출 이미지:", len(submit_paths))
assert len(image_paths) > 0 and len(submit_paths) > 0


In [4]:
# ===== STEP 3 : 증강 =====
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.55, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(0.5),
    T.RandomVerticalFlip(0.5),
    T.RandomApply([T.RandomRotation(20)], p=0.5),
    T.RandAugment(num_ops=2, magnitude=9),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.14)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class PaintDataset(Dataset):
    def __init__(self, paths, targets=None, transform=None):
        self.paths = list(paths); self.targets = targets; self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("RGB")
        x = self.transform(img)
        y = -1 if self.targets is None else int(self.targets[i])
        return x, y

def make_loader(paths, targets, tf, shuffle, drop_last=False):
    return DataLoader(PaintDataset(paths, targets, tf), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=USE_GPU, drop_last=drop_last)


In [ ]:
# ===== STEP 4 : 모델 =====
def build_model(name, num_classes=NUM_CLASSES):
    if name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        in_f = m.classifier[2].in_features
        m.classifier[2] = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_f = m.classifier[1].in_features
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
    else:
        raise ValueError(name)
    return m.to(DEVICE)

# 클래스 불균형 보정 가중치
cls_count = np.bincount(Y, minlength=NUM_CLASSES)
cls_w = torch.tensor((cls_count.sum() / (NUM_CLASSES * cls_count)), dtype=torch.float32).to(DEVICE)
print("class weights:", dict(zip(le.classes_, np.round(cls_w.cpu().numpy(), 3))))


In [ ]:
# ===== STEP 5 : 학습 루프 =====
def mixup(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def train_one_model(model_name, tr_p, tr_y, va_p, va_y, tag):
    model = build_model(model_name)
    tr_loader = make_loader(tr_p, tr_y, train_tf, True, drop_last=True)
    va_loader = make_loader(va_p, va_y, eval_tf, False)

    criterion = nn.CrossEntropyLoss(weight=cls_w, label_smoothing=LABEL_SMOOTH)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tr_loader), pct_start=0.25)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    best_f1, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train(); run_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                if random.random() < MIXUP_P:
                    xm, ya, yb2, lam = mixup(xb, yb)
                    out = model(xm)
                    loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb2)
                else:
                    loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            run_loss += loss.item()

        model.eval(); preds = []
        with torch.no_grad():
            for xb, _ in va_loader:
                with torch.cuda.amp.autocast(enabled=AMP):
                    preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
        preds = np.concatenate(preds)
        f1  = f1_score(va_y, preds, average="macro")
        acc = accuracy_score(va_y, preds)
        print(f"[{tag}] ep{ep+1:02d} loss={run_loss/len(tr_loader):.4f} acc={acc:.4f} macroF1={f1:.4f}")
        if f1 >= best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    torch.save(best_state, CKPT_DIR / f"{tag}.pt")
    model.load_state_dict(best_state)
    print(f"[{tag}] BEST macroF1 = {best_f1:.4f}")
    return model, best_f1


In [ ]:
# ===== STEP 6 : TTA 추론 =====
@torch.no_grad()
def predict_proba(model, paths, tta=True):
    loader = make_loader(paths, None, eval_tf, False)
    model.eval(); out = []
    for xb, _ in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        views = [xb, torch.flip(xb,[3]), torch.flip(xb,[2]), torch.flip(xb,[2,3])] if tta else [xb]
        p = 0
        for v in views:
            with torch.cuda.amp.autocast(enabled=AMP):
                p = p + torch.softmax(model(v).float(), dim=1)
        out.append((p / len(views)).cpu().numpy())
    return np.concatenate(out, axis=0)


In [ ]:
# ===== STEP 7 : K-Fold 앙상블 =====
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_prob   = np.zeros((len(image_paths), NUM_CLASSES), dtype=np.float32)
oof_weight = np.zeros(len(image_paths), dtype=np.float32)
test_prob  = np.zeros((len(submit_paths), NUM_CLASSES), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(image_paths, Y)):
    for mname in MODEL_NAMES:
        tag = f"{mname}_fold{fold}"
        seed_everything(SEED + fold)
        model, f1 = train_one_model(mname, image_paths[tr_idx], Y[tr_idx],
                                    image_paths[va_idx],  Y[va_idx], tag)
        w = max(f1, 1e-3)
        oof_prob[va_idx]  += predict_proba(model, image_paths[va_idx]) * w
        oof_weight[va_idx]+= w
        test_prob         += predict_proba(model, submit_paths) * w
        fold_scores.append(f1)
        del model; gc.collect(); torch.cuda.empty_cache()

oof_prob /= oof_weight[:, None]
print("\n개별 모델 평균 macroF1 :", round(float(np.mean(fold_scores)), 4))
print("앙상블 OOF macroF1     :", round(f1_score(Y, oof_prob.argmax(1), average='macro'), 4))
print("앙상블 OOF Accuracy    :", round(accuracy_score(Y, oof_prob.argmax(1)), 4))


In [ ]:
# ===== STEP 8 : 진단 =====
pred_oof = le.inverse_transform(oof_prob.argmax(1))
print(classification_report(labels, pred_oof, digits=4))
cm = confusion_matrix(labels, pred_oof, labels=list(le.classes_))
print(pd.DataFrame(cm, index=le.classes_, columns=le.classes_))

## 학습 방법 / 성능을 더 올리는 방법

### 기본 학습 방법
위 셀을 위에서부터 순서대로 실행하면 됩니다.

1. 학습 이미지 경로 수집
2. ResNet18로 512차원 특징 추출
3. 80% train / 20% validation 분할
4. LogisticRegression 학습
5. `Macro F1`이 가장 높은 모델 선택
6. 제출용 이미지의 특징을 추출한 뒤 예측

**GPU가 있으면 특징 추출이 훨씬 빠릅니다.** LogisticRegression 자체는 CPU에서 학습됩니다.

### 95%에 조금 못 미칠 때 먼저 해볼 것
코드를 크게 바꾸지 않고 아래 순서대로 시도하세요.

- `C_CANDIDATES`에 `30.0` 추가
- validation 결과에서 특정 클래스의 Recall/F1이 낮다면 해당 클래스 데이터 수 확인
- `BATCH_SIZE`는 성능과 무관하며 속도/메모리만 조절함
- 검증 F1이 높지만 리더보드가 낮으면 train/validation 분포와 실제 제출 데이터 분포 차이를 의심

### 더 강하게 학습하고 싶을 때
이 노트북은 **ResNet18을 고정된 특징 추출기**로 사용합니다. 따라서 변경 폭이 작고 학습이 빠릅니다.

그래도 목표 점수에 도달하지 못하면 그 다음 단계는 ResNet18의 마지막 블록(`layer4`)과 분류층을 5~10 epoch 정도 미세조정(fine-tuning)하는 것입니다. 이 방법은 코드 변경량과 학습 시간이 커지므로 이 최소 수정본에는 기본으로 넣지 않았습니다.

## 모델 평가 및 결과 저장

기존 제출 형식은 그대로 유지합니다.

- index: 테스트 이미지 파일명
- `label`: 예측한 한글 라벨


In [ ]:
# ===== STEP 9 : 제출 =====
final_pred = le.inverse_transform(test_prob.argmax(1))
submission = pd.DataFrame({"label": final_pred},
                          index=[Path(p).name for p in submit_paths])
submission.index.name = None
submission.to_csv("submission.csv", encoding="utf-8-sig")

check = pd.read_csv("submission.csv", index_col=0)
assert len(check) == len(submit_paths)
assert set(check["label"]).issubset(set(le.classes_))
print("검증 완료 | 행 수:", len(check))
print(check["label"].value_counts())
print(check.head())

### 제출

`submission.csv` 파일이 정상적으로 생성되면 Elice의 제출 버튼을 이용해 제출하세요.

이 버전은 기존 베이스라인을 크게 바꾸지 않고도 이미지의 색상/형태/질감 정보를 훨씬 잘 활용하도록 수정한 버전입니다.

> **중요:** 검증 Accuracy 또는 Macro F1이 0.95를 넘는다고 실제 리더보드 0.95가 보장되는 것은 아닙니다. 최종 점수는 제출 데이터의 분포와 라벨 품질에도 영향을 받습니다.